In [7]:
import gymnasium as gym
import torch
import torch.optim as opti
import torch.nn as nn
from torch.distributions import Normal
import numpy as np

In [4]:
env = gym.make('Pendulum-v1')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
action_low = float(env.action_space.low[0])
action_high = float(env.action_space.high[0])

In [11]:
from matplotlib.pyplot import axis
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, action_high, hidden_dim = 256):
        super ().__init__()
        self.action_high = action_high
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        self.mean_calc = nn.Linear(128, action_dim)
        self.log_std_calc = nn.Linear(128, action_dim)
    
    def forward(self, state):
        features = self.net(features)
        mean = self.mean_calc(features)
        log_std = self.log_std_calc(features)
        log_std = torch.clamp(log_std, min = -20, max = 5)
        std = torch.exp(log_std)
        return Normal(mean , std)
    
    def get_action(self, state):
        distrib = self.forward(state)
        raw_action = distrib.rsample()
        squashed_action = torch.tanh(raw_action)
        scaled_action = squashed_action * self.action_high
        log_prob = distrib.log_prob(raw_action).sum(axis = -1, keepdim = True)
        log_prob -= torch.log(1 - squashed_action.pow(2) + 1e-6).sum(axis= -1, keepdim = True)
        return scaled_action, log_prob

In [12]:
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU6(),
            nn.Linear(256, 1)
        )
    def forward(self, state, action):
        x = torch.cat([state, action], dim =1)
        return self.net(x)

In [13]:
batch_size = 32
alpha = 0.2
gamma = 0.99